# Multi-Folding Competition Candidates
(using the Boltz API)

In [2]:
import pandas as pd
import os, time
from boltz_api import Boltz

In [3]:
candidates_df = pd.read_csv("../../data/proteinbase_collection_nipah-binder-competition-all-submissions.csv")

candidates_df.head()

,id,name,sequence,author,designMethod,evaluations
0,swift-otter-reed,target_binder_design_cdr3_fixed_run_45_cycle_8...,QVQLVESGGGLVQPGGSLRLSCAASGFSFSYYWLGWFRQAPGQGLE...,willv,cdr3-optimization-with-protein-hunter-ranking-...,"[{""type"":""computational"",""value"":""Mainly Beta""..."
1,azure-fox-flint,target_binder_design_cdr3_fixed_run_14_cycle_8...,QVQLVESGGGLVQPGGSLRLSCAASGFSFSYYWLGWFRQAPGQGLE...,willv,cdr3-optimization-with-protein-hunter-ranking-...,"[{""type"":""computational"",""value"":[{""chain"":""B""..."
2,dark-seal-lava,d26,SKPGEKGPKIVLTPKPGYKVYMVDKDQADRCTIKKENTPLLNAAKP...,sasha-murrell,NaN,"[{""type"":""computational"",""value"":""Few Secondar..."
3,calm-orca-ruby,1,KSTAIKATELQLKLLDALENDAPFEEIVAILRELLELLSDLGAAQL...,tom-pan,bg-top-hits-w8qr9efM38,"[{""type"":""computational"",""unit"":""%"",""value"":87..."
4,noble-lynx-lotus,5,SATVTLTALSDFEITVTVTGEGVKEVDVFTASAVDCGFERIKAGGS...,tom-pan,bg-top-hits-w8qr9efM38,"[{""type"":""computational"",""unit"":""%"",""value"":0...."


In [3]:
antigen_seq = "QNYTRSTDNQAVIKDALQGIQQQIKGLADKIGTEIGPKVSLIDTSSTITIPANIGLLGSKISQSTASINENVNEKCKFTLPPLKIHECNISCPNPLPFREYRPQTEGVSNLVGLPNNICLQKTSNQILKPKLISYTLPVVGQSGTCITDPLLAMDEGYFAYSHLERIGSCSRGVSKQRIIGVGEVLDRGDEVPSLFMTNVWTPPNPNTVYHCSAVYNNEFYYVLCAVSTVGDPILNSTYWSGSLMMTRLAVKPKSNGGGYNQHQLALRSIEKGRYDKVMPYGPSGIKQGDTLYFPAVGFLVRTEFKYNDSNCPITKCQYSKPENCRLSMGIRPNSHYILRSGLLKYNLSDGENPKVVFIEISDQRLSIGSPSKIYDSLGQPVFYQASFSWDTMIKFGDVLTVNPLVVNWRNNTVISRPGQSQCPRFNTCPEICWEGVYNDAFLIDRINWISAGVFLDSNQTAENPVFTVFKDNEILYRAQLASEDTNAQKTITNCFLLKNKIWCISLVEIYDTGDNVIRPKLFAVKIPEQCT"

## Run Screen
Source: https://api.boltz.bio/docs/guides/protein-library-screen/

In [5]:
client = Boltz(api_key=os.environ["BOLTZ_API_KEY"])

In [6]:
target = {
    "type": "no_template",
    "entities": [
        {
            "type": "protein",
            "value": antigen_seq,
            "chain_ids": ["A"]
        }
    ]
}

In [7]:
proteins = []

for i, row in candidates_df.iterrows():
    submission_seq = row['sequence']
    competition_id = row["id"]
    proteins.append(
        {
            "entities": [
                {"type": "protein", "value": submission_seq, "chain_ids": ["H"]},
            ], "id": competition_id
        }
    )


In [8]:
## Submit now and download later:
screen = client.protein.library_screen.start(target=target, proteins=proteins)

In [9]:
## Print Screen ID
print(f"Screen ID: {screen.id}")

Screen ID: prot_scr_Fz18T6T9nEvYcTBV2wxx


In [10]:
## Poll the run for status and progress.
while screen.status not in ("succeeded", "failed", "stopped"):
    time.sleep(10)
    screen = client.protein.library_screen.retrieve(screen.id)
    p = screen.progress
    print(f"{screen.status}: {p.num_proteins_screened}/{p.total_proteins_to_screen}")


running: 0/3749
running: 0/3749
running: 0/3749
running: 4/3749
running: 6/3749
running: 6/3749
running: 10/3749
running: 12/3749
running: 13/3749
running: 16/3749
running: 18/3749
running: 22/3749
running: 25/3749
running: 32/3749
running: 39/3749
running: 49/3749
running: 58/3749
running: 69/3749
running: 84/3749
running: 95/3749
running: 107/3749
running: 124/3749
running: 140/3749
running: 163/3749
running: 190/3749
running: 211/3749
running: 230/3749
running: 250/3749
running: 257/3749
running: 263/3749
running: 267/3749
running: 272/3749
running: 284/3749
running: 304/3749
running: 328/3749
running: 379/3749
running: 404/3749
running: 430/3749
running: 472/3749
running: 502/3749
running: 527/3749
running: 553/3749
running: 589/3749
running: 622/3749
running: 658/3749
running: 688/3749
running: 725/3749
running: 755/3749
running: 783/3749
running: 814/3749
running: 850/3749
running: 878/3749
running: 911/3749
running: 949/3749
running: 979/3749
running: 1010/3749
running: 1047/374

In [11]:
## Best first: highest binding confidence, then lowest interface error.
## Use external_id to correlate each result back to the protein you submitted.
results = list(client.protein.library_screen.list_results(screen.id))
results.sort(key=lambda r: (-r.metrics.binding_confidence, r.metrics.min_interaction_pae))
for r in results[:5]:
    print(
        f"{r.id}  "
        f"ext={r.external_id}  "
        f"bind={r.metrics.binding_confidence:.2f}  "
        f"structure_confidence={r.metrics.structure_confidence:.2f}  "
        f"iPAE={r.metrics.min_interaction_pae:.1f}Å  "
    )

pres_DGJWO4imCGvUWnsrmGuK  ext=hollow-heron-thorn  bind=0.77  structure_confidence=0.80  iPAE=0.9Å  
pres_h5IpFLfIaKoeparbvHby  ext=deep-crane-ruby  bind=0.76  structure_confidence=0.74  iPAE=1.6Å  
pres_DqLPBr4ayLjBGxbzaCBe  ext=jade-bear-willow  bind=0.76  structure_confidence=0.77  iPAE=1.0Å  
pres_eBEwe54PLOuipyGc0vOB  ext=scarlet-boar-quartz  bind=0.76  structure_confidence=0.73  iPAE=1.3Å  
pres_HDb4iNfHAecAls1XeYnB  ext=frozen-raven-maple  bind=0.76  structure_confidence=0.72  iPAE=1.2Å  


In [12]:
## Write results to pickle file
import pickle
with open("screen_results.pkl", "wb") as f:
    pickle.dump(results, f)

In [ ]:
# results = pickle.load(open("screen_results.pkl", "rb"))

In [13]:
def parse_metrics(result):
    return {
        "id": result.id,
        "external_id": result.external_id,
        "created_at": result.created_at,
        "archive_url": result.artifacts.archive.url,
        "archive_url_expires_at": result.artifacts.archive.url_expires_at,
        "structure_url": result.artifacts.structure.url,
        "structure_url_expires_at": result.artifacts.structure.url_expires_at,
        "entities": [e.to_dict() for e in result.entities],
        "binding_confidence": result.metrics.binding_confidence,
        "helix_fraction": result.metrics.helix_fraction,
        "iptm": result.metrics.iptm,
        "loop_fraction": result.metrics.loop_fraction,
        "min_interaction_pae": result.metrics.min_interaction_pae,
        "sheet_fraction": result.metrics.sheet_fraction,
        "structure_confidence": result.metrics.structure_confidence
    }

In [14]:
results_dict = [parse_metrics(results[i]) for i in range(len(results))]

In [15]:
results_df = pd.DataFrame(results_dict)

results_df.head()

,id,external_id,created_at,archive_url,archive_url_expires_at,structure_url,structure_url_expires_at,entities,binding_confidence,helix_fraction,iptm,loop_fraction,min_interaction_pae,sheet_fraction,structure_confidence
0,pres_DGJWO4imCGvUWnsrmGuK,hollow-heron-thorn,2026-06-14 16:23:32.331000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 17:00:01.258000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 17:00:01.237000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.773935,0.493151,0.952705,0.232877,0.887306,0.273973,0.803754
1,pres_h5IpFLfIaKoeparbvHby,deep-crane-ruby,2026-06-14 16:27:44.250000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 17:00:02.063000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 17:00:02.044000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.764130,0.847619,0.941551,0.152381,1.601647,0.000000,0.735776
2,pres_DqLPBr4ayLjBGxbzaCBe,jade-bear-willow,2026-06-14 16:08:19.311000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:59:57.928000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 16:59:57.906000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.762484,0.277778,0.950799,0.377778,1.046058,0.344444,0.773669
3,pres_eBEwe54PLOuipyGc0vOB,scarlet-boar-quartz,2026-06-14 16:27:49.250000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 17:00:02.229000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 17:00:02.210000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.760238,0.926829,0.938958,0.073171,1.300307,0.000000,0.730751
4,pres_HDb4iNfHAecAls1XeYnB,frozen-raven-maple,2026-06-14 16:28:16.160000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 17:00:02.374000+00:00,https://boltz-platform-prod-compute-api-storag...,2026-06-14 17:00:02.356000+00:00,"[{'chain_ids': ['H'], 'type': 'protein', 'valu...",0.760215,0.902439,0.941366,0.097561,1.197569,0.000000,0.716424


In [16]:
results_df.to_csv("screen_results.csv", index=False)

In [ ]:
## Stop the Screen
# client.protein.library_screen.stop(screen.id)

## Combine with Experimental Results

In [12]:
binding_results_df = pd.read_csv("../../data/proteinbase_collection_nipah-binder-competition-results_processed.csv")

binding_cols_to_keep = ["id", "binding", "binding_strength", "kd", "kon", "koff", "neutralization"]
binding_results_df = binding_results_df[binding_cols_to_keep]


binding_results_df.head()

,id,binding,binding_strength,kd,kon,koff,neutralization
0,azure-wolf-maple,True,Strong,2.456564e-09,51499.707222,0.000199,NaN
1,calm-panda-fern,True,Strong,5.118656e-09,44544.734509,0.000445,NaN
2,deep-heron-rose,True,Strong,8.760552e-10,815100.415700,0.000704,NaN
3,ivory-orca-fern,True,Strong,5.204169e-09,8228.269630,0.000090,NaN
4,azure-fox-flint,False,NaN,NaN,NaN,NaN,NaN


In [ ]:
results_df = pd.read_csv("screen_results.csv")[["external_id", "binding_confidence", "iptm", "min_interaction_pae", "structure_confidence"]]

final_df = binding_results_df.merge(results_df, left_on="id", right_on="external_id", how="left").drop(columns=['external_id'])

final_df.head()

,id,binding,binding_strength,kd,kon,koff,neutralization,binding_confidence,iptm,min_interaction_pae,structure_confidence
0,azure-wolf-maple,True,Strong,2.456564e-09,51499.707222,0.000199,NaN,NaN,NaN,NaN,NaN
1,calm-panda-fern,True,Strong,5.118656e-09,44544.734509,0.000445,NaN,NaN,NaN,NaN,NaN
2,deep-heron-rose,True,Strong,8.760552e-10,815100.415700,0.000704,NaN,NaN,NaN,NaN,NaN
3,ivory-orca-fern,True,Strong,5.204169e-09,8228.269630,0.000090,NaN,NaN,NaN,NaN,NaN
4,azure-fox-flint,False,NaN,NaN,NaN,NaN,NaN,0.000009,0.307230,15.836696,0.043041
...,...,...,...,...,...,...,...,...,...,...,...
1025,quick-boar-fern,False,NaN,NaN,NaN,NaN,NaN,0.528906,0.883870,2.509221,0.757235
1026,lunar-quail-dust,True,Weak,NaN,NaN,NaN,NaN,0.352256,0.620794,9.443636,0.196013
1027,hollow-zebra-crystal,False,NaN,NaN,NaN,NaN,NaN,0.000701,0.873958,3.034688,0.695581
1028,ivory-orca-ivy,True,Medium,4.959858e-07,4286.175021,0.002109,NaN,0.000793,0.393334,17.645906,0.015690


In [18]:
final_results = final_df.to_csv("final_results.csv", index=False)